Copias impresas y electrónicas de *Modelado y simulación en Python* están disponibles en [No Starch Press](https://nostarch.com/modeling-and-simulation-python) y [Bookshop.org](https://bookshop.org/p/books/modeling-and-simulation-in-python-allen-b-downey/17836697?ean=9781718502161) y [Amazon](https://amzn.to/3y9UxNb).

# Límites al crecimiento

*Modelado y Simulación en Python*

Copyright 2021 Allen Downey

Licencia: [Creative Commons Atribución-No Comercial-CompartirIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [1]:
# download modsim.py if necessary

from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve
        local, _ = urlretrieve(url, filename)
        print('Downloaded ' + local)
    
download('https://github.com/AllenDowney/ModSimPy/raw/master/' +
         'modsim.py')

In [2]:
# import functions from modsim

from modsim import *

Este capítulo está disponible como un cuaderno Jupyter donde puede leer el texto, ejecutar el código y trabajar en los ejercicios. 
Haga clic aquí para acceder a los cuadernos: <https://allendowney.github.io/ModSimPy/>.

Aquí tenéis de nuevo los datos del capítulo anterior.

In [3]:
download('https://raw.githubusercontent.com/AllenDowney/' +
         'ModSimPy/master/data/World_population_estimates.html')

In [4]:
from pandas import read_html

filename = 'World_population_estimates.html'
tables = read_html(filename, header=0, index_col=0, decimal='M')
table2 = tables[2]
table2.columns = ['census', 'prb', 'un', 'maddison', 
                  'hyde', 'tanton', 'biraben', 'mj', 
                  'thomlinson', 'durand', 'clark']

In [5]:
un = table2.un / 1e9
census = table2.census / 1e9

Y aquí están las funciones del capítulo anterior.

In [6]:
download('https://github.com/AllenDowney/ModSimPy/raw/master/' +
         'chap06.py')

In [7]:
from chap06 import run_simulation

def plot_estimates():
    census.plot(style=':', label='US Census')
    un.plot(style='--', label='UN DESA')
    decorate(xlabel='Year', 
             ylabel='World population (billions)') 

En el capítulo anterior desarrollamos un modelo de población donde el crecimiento neto durante cada paso de tiempo es proporcional a la población actual. Este modelo parece más realista que el modelo de crecimiento constante, pero no se ajusta tan bien a los datos.

Hay algunas cosas que podríamos intentar mejorar el modelo:

- Quizás el crecimiento neto dependa de la población actual, pero el
    La relación es cuadrática, no lineal.

- Quizás la tasa de crecimiento neto varíe con el tiempo.

En este capítulo, exploraremos la primera opción.
En los ejercicios tendrás la oportunidad de probar el segundo. 

## Crecimiento cuadrático

Tiene sentido que el crecimiento neto dependa de la población actual, pero tal vez no sea una relación lineal, como esta:

```
net_growth = system.alpha * pop
```

Quizás sea una relación cuadrática, como esta:

```
net_growth = system.alpha * pop + system.beta * pop**2
```

Podemos probar esa conjetura con una nueva función de actualización:

In [8]:
def growth_func_quad(t, pop, system):
    return system.alpha * pop + system.beta * pop**2

Aquí está el objeto `System` que usaremos, inicializado con `t_0`, `p_0` y `t_end`.

In [9]:
t_0 = census.index[0]
p_0 = census[t_0]
t_end = census.index[-1]

system = System(t_0=t_0,
                p_0=p_0,
                t_end=t_end)

.
Ahora tenemos que agregar los parámetros `alpha` y `beta`.
Elegí los siguientes valores por prueba y error; veremos mejores formas de hacerlo más adelante.

In [10]:
system.alpha = 25 / 1000
system.beta = -1.8 / 1000

Y así es como lo ejecutamos: 

In [11]:
results = run_simulation(system, growth_func_quad)

Aquí están los resultados.

In [12]:
results.plot(color='gray', label='model')
plot_estimates()
decorate(title='Quadratic growth model')

El modelo se ajusta bien a los datos en todo el rango, con sólo un poco de espacio entre ellos en la década de 1960.

No es del todo sorprendente que el modelo cuadrático se ajuste mejor que el
modelos constante y proporcional, debido a que tiene dos parámetros podemos
elegir, donde los otros modelos solo tienen uno. En general, cuanto más
parámetros con los que tienes que jugar, mejor deberías esperar que sea el modelo
para encajar.

Pero ajustar los datos no es la única razón para pensar en el modelo cuadrático.
podría ser una buena elección. También tiene sentido; es decir, hay un
razón legítima para esperar que la relación entre crecimiento y
población tenga esta forma.

Para entenderlo, veamos el crecimiento neto en función de la población.

## Crecimiento neto

Tracemos la relación entre crecimiento y población en el modelo cuadrático.
Usaré `linspace` para crear una matriz de 101 poblaciones de 0 a 15 mil millones.

In [13]:
from numpy import linspace

pop_array = linspace(0, 15, 101)

Ahora usaré el modelo cuadrático para calcular el crecimiento neto de cada población.

In [14]:
growth_array = (system.alpha * pop_array + 
                system.beta * pop_array**2)

Para trazar la tasa de crecimiento versus la población, usaremos la función `plot` de Matplotlib.
Primero tenemos que importarlo:

In [15]:
from matplotlib.pyplot import plot

Ahora podemos usarlo así:

In [16]:
plot(pop_array, growth_array, label='net growth', color='C2')

decorate(xlabel='Population (billions)',
         ylabel='Net growth (billions)',
         title='Net growth vs. population')

Tenga en cuenta que el eje x no es el tiempo, como en las figuras anteriores, sino la población. Podemos dividir esta curva en cuatro tipos de comportamiento:

- Cuando la población es inferior a 3 mil millones de habitantes, el crecimiento neto es
    proporcional a la población, como en el modelo proporcional. en esto
    rango, la población crece lentamente porque la población es pequeña.

- Entre 3 mil millones y 10 mil millones, la población crece rápidamente
    porque hay mucha gente.

- Por encima de los 10 mil millones, la población crece más lentamente; este comportamiento modela
    el efecto de las limitaciones de recursos que disminuyen las tasas de natalidad o
    aumentar las tasas de mortalidad.

- Por encima de 14 mil millones, los recursos son tan limitados que la tasa de mortalidad
    supera la tasa de natalidad y el crecimiento neto se vuelve negativo.

Justo debajo de los 14 mil millones, hay un punto en el que el crecimiento neto es 0, lo que
significa que la población no cambia. En este punto, las tasas de natalidad y mortalidad son iguales, por lo que la población está en *equilibrio*.

## Encontrar el equilibrio

El punto de equilibrio es la población, $p$, donde el crecimiento neto de la población, $\Delta p$, es 0.
Podemos calcularlo encontrando las raíces, o ceros, de esta ecuación: 

$$\Delta p = \alpha p + \beta p^2$$ 

donde $\alpha$ y $\beta$ son los parámetros del modelo. 
Si reescribimos el lado derecho así: 

$$\Delta p = p (\alpha + \beta p)$$ 

podemos ver que el crecimiento neto es $0$ cuando $p=0$ o $p=-\alpha/\beta$.
Entonces podemos calcular el punto de equilibrio (distinto de cero) así:

In [17]:
-system.alpha / system.beta

Con estos parámetros, el crecimiento neto es 0 cuando la población es de aproximadamente 13,9 mil millones.
(el resultado es positivo porque `beta` es negativo).

En el contexto del modelado poblacional, el modelo cuadrático es más
escrito convencionalmente así: 

$$\Delta p = r p (1 - p / K)$$ 

Este es el mismo modelo; es simplemente una forma diferente de *parametrizarlo*. Dados $\alpha$ y $\beta$, podemos calcular $r=\alpha$ y $K=-\alpha/\beta$.

En esta versión, es más fácil interpretar los parámetros: $r$ es el
tasa de crecimiento sin restricciones, observada cuando $p$ es pequeño y $K$ es el
punto de equilibrio. 
$K$ también se llama *capacidad de carga*, ya que indica la población máxima que el medio ambiente puede sostener.

## Resumen

En este capítulo implementamos un modelo de crecimiento cuadrático donde el crecimiento neto depende de la población actual y de la población al cuadrado.
Este modelo se ajusta bien a los datos y vimos una razón: se basa en el supuesto de que existe un límite en la cantidad de personas que la Tierra puede sustentar.

En el próximo capítulo usaremos los modelos que hemos desarrollado para generar
predicciones
Pero primero, quiero advertirte sobre algunas cosas que pueden salir mal al escribir funciones.

## Disfunciones

Cuando las personas aprenden sobre funciones, hay algunas cosas que a menudo
encontrar confuso. En esta sección presentaré y explicaré algunos de los más comunes.
problemas.

Como ejemplo, supongamos que desea una función que tome un
Objeto `System`, con las variables `alpha` y `beta`, y calcula el
capacidad de carga, `-alpha/beta`. 
Aquí tienes una buena solución: 

In [18]:
def carrying_capacity(system):
    K = -system.alpha / system.beta
    return K
    
sys1 = System(alpha=0.025, beta=-0.0018)
pop = carrying_capacity(sys1)
print(pop)

Ahora veamos todas las formas en que pueden salir mal.

*Disfunción #1:* No usar parámetros. En la siguiente versión, la función no toma ningún parámetro; cuando `sys1` aparece dentro de la función, se refiere al objeto que creamos fuera de la función.

In [19]:
def carrying_capacity():
    K = -sys1.alpha / sys1.beta
    return K
    
sys1 = System(alpha=0.025, beta=-0.0018)
pop = carrying_capacity()
print(pop)

Esta versión funciona, pero no es tan versátil como podría ser.
Si hay varios objetos `System`, esta función puede funcionar solo con uno de ellos, y solo si se llama `sys1`.

*Disfunción #2:* Golpear los parámetros. Cuando la gente aprende por primera vez
sobre parámetros, a menudo escriben funciones como esta:

In [20]:
# WRONG
def carrying_capacity(system):
    system = System(alpha=0.025, beta=-0.0018)
    K = -system.alpha / system.beta
    return K
    
sys1 = System(alpha=0.03, beta=-0.002)
pop = carrying_capacity(sys1)
print(pop)

En este ejemplo, tenemos un objeto `System` llamado `sys1` que se pasa
como argumento para `carrying_capacity`. Pero cuando la función se ejecuta,
ignora el argumento y lo reemplaza inmediatamente con un nuevo `System`
objeto. Como resultado, esta función siempre devuelve el mismo valor, no
importa qué argumento se pase.

Cuando escribes una función, generalmente no sabes cuáles son los valores de
serán los parámetros. Su trabajo es escribir una función que funcione para
cualquier valor válido. Si asigna sus propios valores a los parámetros,
derrotar todo el propósito de las funciones.

*Disfunción n.º 3:* Sin valor de retorno. Aquí hay una versión que calcula el valor de `K` pero no lo devuelve.

In [21]:
# WRONG
def carrying_capacity(system):
    K = -system.alpha / system.beta
    
sys1 = System(alpha=0.025, beta=-0.0018)
pop = carrying_capacity(sys1)
print(pop)

Una función que no tiene una declaración de retorno en realidad devuelve un valor especial llamado `None`, por lo que en este ejemplo el valor de `pop` es `None`. Si está depurando un programa y descubre que el valor de una variable es `None` cuando no debería serlo, una función sin una declaración de retorno es una causa probable.

*Disfunción n.º 4:* Ignorar el valor de retorno. Finalmente, aquí hay una versión donde la función es correcta, pero la forma en que se usa no lo es.

```
def carrying_capacity(system):
    K = -system.alpha / system.beta
    return K
    
sys1 = System(alpha=0.025, beta=-0.0018)
carrying_capacity(sys1)   # WRONG
print(K)
```

En este ejemplo, `carrying_capacity` se ejecuta y devuelve `K`, pero el
El valor de retorno no se muestra ni se asigna a una variable.
Si intentamos imprimir `K`, obtenemos un `NameError`, porque `K` solo existe dentro de la función.

Cuando llamas a una función que devuelve un valor, debes hacer algo
con el resultado.

## Ejercicios

### Ejercicio 1

 En una sección anterior vimos una forma diferente de parametrizar el modelo cuadrático:

$$ \Delta p = r p (1 - p / K) $$

donde $r=\alpha$ y $K=-\alpha/\beta$.  

Escriba una versión de `growth_func` que implemente esta versión del modelo.  Pruébelo calculando los valores de `r` y `K` que corresponden a `alpha=0.025` y `beta=-0.0018`, y confirme que obtiene los mismos resultados. 

In [22]:
# Solution goes here

In [23]:
# Solution goes here

In [24]:
# Solution goes here

### Ejercicio 2

  ¿Qué sucede si comenzamos con una población inicial por encima de la capacidad de carga, como 20 mil millones?  Ejecute el modelo con poblaciones iniciales entre 1 y 20 mil millones y represente los resultados en los mismos ejes.

Sugerencia: si hay demasiadas etiquetas en la leyenda, puede trazar resultados como este:

```
    results.plot(label='_nolegend')
```

In [25]:
# Solution goes here